# LeetCode #151 — Reverse Words in a String: đảo thứ tự các từ

**Mức độ:** Medium · **Chủ đề:** Two Pointers, String

Skeleton trên LeetCode:

```python
class Solution:
    def reverseWords(self, s: str) -> str:
```

Mục tiêu của notebook:

1. Cách Pythonic: `" ".join(reversed(s.split()))` — và vì sao `split()` **không tham số** xử lý được khoảng trắng thừa.
2. Tự tách từ bằng **hai con trỏ** (khi không được dùng `split`).
3. Follow-up `O(1)` bộ nhớ phụ (trên mảng ký tự có thể sửa): **đảo cả chuỗi, rồi đảo từng từ**, kèm dọn khoảng trắng tại chỗ.

# 1. Bài toán là gì?

Cho chuỗi `s`, đảo **thứ tự các từ**. Một từ là dãy ký tự không phải dấu cách. Các từ cách nhau bởi ít nhất một dấu cách.

Kết quả: các từ theo thứ tự ngược, cách nhau **đúng một** dấu cách, **không** có dấu cách ở đầu/cuối.

**Constraints (theo đề):**

- `1 <= s.length <= 10^4`
- `s` gồm chữ cái Anh (hoa, thường), chữ số và dấu cách `' '`.
- Có **ít nhất một** từ trong `s`.

**Follow-up (theo đề):** nếu chuỗi có thể sửa được (mutable) trong ngôn ngữ của bạn, giải tại chỗ với `O(1)` bộ nhớ phụ?

---

## Ví dụ

```text
"the sky is blue"       ->  "blue is sky the"
"  hello world  "       ->  "world hello"
"a good   example"      ->  "example good a"
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi (string) trong Python
2. Hai con trỏ (two pointers)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [1]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [2]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [3]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [4]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [5]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [6]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Áp dụng vào bài này

Kiến thức về **chuỗi** Python: `split()` không tham số bỏ được khoảng trắng thừa, `join` để ghép. Bản tự làm: duyệt từ cuối bằng **con trỏ** để cắt từng từ; bản tại chỗ: **đảo cả chuỗi** rồi **đảo từng từ** (đảo đoạn bằng hai con trỏ) và dọn khoảng trắng bằng con trỏ đọc/ghi.

# 3. Nền tảng Python cần biết

## `split()` khác `split(" ")`

In [7]:
from typing import List
import random, time

s = "  a good   example "
print("s.split()    :", s.split())
print('s.split(" ") :', s.split(" "))

s.split()    : ['a', 'good', 'example']
s.split(" ") : ['', '', 'a', 'good', '', '', 'example', '']


- `split()` (không tham số): tách theo **mọi dãy khoảng trắng**, bỏ phần rỗng ở đầu/cuối.
- `split(" ")`: tách tại **từng** dấu cách → sinh ra các chuỗi rỗng `''`.

## Chuỗi Python bất biến

`s[i] = 'x'` lỗi. Muốn sửa tại chỗ phải đổi sang `list(s)`, sửa, rồi `"".join(...)`. Vì thế follow-up `O(1)` chỉ có ý nghĩa thật trong C/C++ — ở Python ta mô phỏng trên `list`.

# 4. Cách 1: Pythonic

In [8]:
def reverse_words_pythonic(s: str) -> str:
    return " ".join(reversed(s.split()))

for t in ["the sky is blue", "  hello world  ", "a good   example"]:
    print(repr(t), "->", repr(reverse_words_pythonic(t)))

'the sky is blue' -> 'blue is sky the'
'  hello world  ' -> 'world hello'
'a good   example' -> 'example good a'


`O(n)` thời gian và bộ nhớ. Trong phỏng vấn, người hỏi thường muốn thấy bạn **tự tách từ** hoặc làm follow-up.

# 5. Cách 2: tự tách từ bằng hai con trỏ (duyệt từ cuối)

Duyệt từ **cuối** về đầu: bỏ qua dấu cách, đánh dấu cuối từ `j`, lùi tới đầu từ `i`, lấy `s[i+1 : j+1]`. Các từ được thu theo đúng thứ tự cần in.

In [9]:
def reverse_words_two_pointers(s: str) -> str:
    words = []
    i = len(s) - 1
    while i >= 0:
        while i >= 0 and s[i] == " ":     # bỏ dấu cách
            i -= 1
        if i < 0:
            break
        j = i                              # cuối từ
        while i >= 0 and s[i] != " ":      # lùi tới trước đầu từ
            i -= 1
        words.append(s[i + 1:j + 1])
    return " ".join(words)

for t in ["the sky is blue", "  hello world  ", "a good   example"]:
    print(repr(t), "->", repr(reverse_words_two_pointers(t)))

'the sky is blue' -> 'blue is sky the'
'  hello world  ' -> 'world hello'
'a good   example' -> 'example good a'


# 6. Ý tưởng chính (follow-up): đảo cả chuỗi, rồi đảo từng từ

Quan sát với `"the sky is blue"`:

```text
đảo cả chuỗi:   "eulb si yks eht"
đảo từng từ:    "blue is sky the"   ✓
```

Đảo cả chuỗi đưa các từ về **đúng vị trí** nhưng mỗi từ bị viết ngược; đảo lại từng từ là xong.

Thêm bước **dọn khoảng trắng tại chỗ** (bỏ đầu/cuối, gộp dấu cách liên tiếp) bằng con trỏ ghi `w`:

```text
1) dọn: chép các ký tự cần giữ về đầu mảng, cắt đuôi
2) đảo toàn bộ a[0:n]
3) đảo từng từ
```

In [10]:
def reverse_range(a: List[str], i: int, j: int) -> None:
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1
        j -= 1


def clean_spaces(a: List[str]) -> int:
    """Dọn tại chỗ; trả về độ dài mới. Giữ 1 dấu cách giữa các từ."""
    w = 0
    for r in range(len(a)):
        if a[r] != " ":
            a[w] = a[r]
            w += 1
        elif w > 0 and a[w - 1] != " ":        # chỉ giữ dấu cách nếu ký tự vừa ghi không phải dấu cách
            a[w] = " "
            w += 1
    if w > 0 and a[w - 1] == " ":             # bỏ dấu cách cuối
        w -= 1
    return w


def reverse_words_inplace(chars: List[str]) -> int:
    """Sửa chars tại chỗ; trả về độ dài kết quả (kết quả là chars[:n])."""
    n = clean_spaces(chars)
    reverse_range(chars, 0, n - 1)
    start = 0
    for k in range(n + 1):
        if k == n or chars[k] == " ":
            reverse_range(chars, start, k - 1)
            start = k + 1
    return n


a = list("  a good   example ")
n = reverse_words_inplace(a)
print(repr("".join(a[:n])))

'example good a'

# 7. Mô phỏng

In [11]:
def trace_inplace(s):
    a = list(s)
    n = clean_spaces(a)
    print(f"ban đầu    : {s!r}")
    print(f"sau dọn    : {''.join(a[:n])!r}  (n = {n})")
    reverse_range(a, 0, n - 1)
    print(f"đảo cả     : {''.join(a[:n])!r}")
    start = 0
    for k in range(n + 1):
        if k == n or a[k] == " ":
            reverse_range(a, start, k - 1)
            print(f"đảo [{start}:{k}] : {''.join(a[:n])!r}")
            start = k + 1
    return "".join(a[:n])

_ = trace_inplace("  the sky  is blue ")

ban đầu    : '  the sky  is blue '
sau dọn    : 'the sky is blue'  (n = 15)
đảo cả     : 'eulb si yks eht'
đảo [0:4] : 'blue si yks eht'
đảo [5:7] : 'blue is yks eht'
đảo [8:11] : 'blue is sky eht'
đảo [12:15] : 'blue is sky the'


# 8. Phiên bản đúng format LeetCode

LeetCode Python nhận `str` (bất biến) nên ta dùng cách 2 — rõ ràng, không phụ thuộc `split`. (Cách 1 cũng được chấp nhận.)

In [12]:
class Solution:
    def reverseWords(self, s: str) -> str:
        words = []
        i = len(s) - 1
        while i >= 0:
            while i >= 0 and s[i] == " ":
                i -= 1
            if i < 0:
                break
            j = i
            while i >= 0 and s[i] != " ":
                i -= 1
            words.append(s[i + 1:j + 1])
        return " ".join(words)

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual!r}")


def inplace_wrapper(s):
    a = list(s)
    n = reverse_words_inplace(a)
    return "".join(a[:n])


sol = Solution()
check("Ví dụ 1", sol.reverseWords("the sky is blue"), "blue is sky the")
check("Ví dụ 2", sol.reverseWords("  hello world  "), "world hello")
check("Ví dụ 3", sol.reverseWords("a good   example"), "example good a")
check("Một từ", sol.reverseWords("   x   "), "x")
check("Không cách", sol.reverseWords("abc"), "abc")

random.seed(151)
for _ in range(5000):
    s = "".join(random.choice("ab1 ") for _ in range(random.randint(1, 20)))
    if not s.strip():
        s = s + "z"
    exp = reverse_words_pythonic(s)
    assert sol.reverseWords(s) == exp == inplace_wrapper(s), repr(s)
print("5000 chuỗi ngẫu nhiên: 3 cách khớp nhau")

big = " ".join("w" * random.randint(1, 8) for _ in range(2000))[:10**4].strip()
t0 = time.time(); sol.reverseWords(big); inplace_wrapper(big); t1 = time.time()
print(f"len = {len(big)}: {t1 - t0:.4f}s")

Ví dụ 1          PASS  'blue is sky the'
Ví dụ 2          PASS  'world hello'
Ví dụ 3          PASS  'example good a'
Một từ           PASS  'x'
Không cách       PASS  'abc'


5000 chuỗi ngẫu nhiên: 3 cách khớp nhau
len = 10000: 0.0111s


# 9. Vì sao đúng? Độ phức tạp?

**Cách 2:** mỗi lượt vòng ngoài bỏ qua một dãy dấu cách rồi lấy trọn **một** từ (dãy ký tự không phải dấu cách liên tiếp, tối đại). Duyệt từ cuối nên từ cuối được lấy trước — đúng thứ tự đảo.

**Cách đảo hai lần:** gọi các từ `w1 w2 … wk`. Đảo cả chuỗi được `rev(wk) … rev(w2) rev(w1)` (vì `rev(XY) = rev(Y) rev(X)`). Đảo lại từng khối được `wk … w2 w1`.

```text
Cách 1 / 2:      Time O(n), bộ nhớ O(n)
Đảo hai lần:     Time O(n), bộ nhớ phụ O(1) (trên mảng có thể sửa)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: `split(" ")`

In [14]:
def wrong_split_space(s):
    return " ".join(reversed(s.split(" ")))

print(repr(wrong_split_space("  hello world  ")), "| đúng:", repr(sol.reverseWords("  hello world  ")))

'  world hello  ' | đúng: 'world hello'


Các chuỗi rỗng `''` sinh ra từ dấu cách thừa được nối lại thành dấu cách ở đầu/cuối (với `"a good   example"` sẽ còn thừa cả dấu cách ở giữa: `'example   good a'`).

## Lỗi 2: Đảo cả chuỗi mà quên đảo từng từ (hoặc ngược lại)

In [15]:
print(repr("the sky is blue"[::-1]), "| chỉ đảo từng từ:", repr(" ".join(w[::-1] for w in "the sky is blue".split())))

'eulb si yks eht' | chỉ đảo từng từ: 'eht yks si eulb'


Đảo cả chuỗi chỉ đảo **ký tự**; đảo từng từ giữ thứ tự từ (đó là bài **#557**). Cần **cả hai** bước.

## Lỗi 3: Dọn khoảng trắng nhưng để sót dấu cách cuối

In [16]:
def wrong_clean(a):
    w = 0
    for r in range(len(a)):
        if a[r] != " ":
            a[w] = a[r]; w += 1
        elif w > 0 and a[w - 1] != " ":
            a[w] = " "; w += 1
    return w                                       # SAI: thiếu bước bỏ dấu cách cuối

a = list("hello world  ")
n = wrong_clean(a)
print(repr("".join(a[:n])))

'hello world '


Sau khi đảo cả chuỗi, dấu cách cuối này nhảy lên **đầu** kết quả.

## Lỗi 4: Nối chuỗi bằng `+=` trong vòng lặp

Mỗi `+=` có thể tạo chuỗi mới → `O(n²)` trường hợp xấu. Gom vào list rồi `" ".join`.

# 11. Mẹo nhận diện pattern

- **Đảo thứ tự các khối, giữ nguyên nội dung mỗi khối** → đảo toàn bộ rồi đảo từng khối (cũng là cách **xoay mảng** #189: đảo cả, đảo hai phần).
- **Chuẩn hóa khoảng trắng tại chỗ** → con trỏ đọc `r`, con trỏ ghi `w` (giống #26, #27, #283).

Bài liên quan: **#186 Reverse Words in a String II** (mảng ký tự, tại chỗ), **#557 Reverse Words in a String III**, **#189 Rotate Array**, **#58 Length of Last Word**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `"  Bob    Loves  Alice   "`, viết kết quả sau bước dọn, sau đảo cả, sau đảo từng từ.

## Bài 2 — Điền chỗ trống (duyệt từ cuối)

In [17]:
def exercise_2_reverse_words(s: str) -> str:
    words = []
    i = len(s) - 1
    while i >= 0:
        # TODO: bỏ dấu cách; nếu hết thì break; đánh dấu j; lùi qua từ; thêm s[i+1:j+1]
        pass
    return " ".join(words)


# assert exercise_2_reverse_words("the sky is blue") == "blue is sky the"
# assert exercise_2_reverse_words("  hello world  ") == "world hello"
# assert exercise_2_reverse_words("a good   example") == "example good a"

## Bài 3 — Biến thể: Reverse Words in a String II (LeetCode #186)

Cho **mảng ký tự** (không có khoảng trắng thừa: không ở đầu/cuối, đúng 1 dấu cách giữa các từ). Đảo thứ tự từ **tại chỗ**, không trả về gì.

In [18]:
def exercise_3_reverse_words_ii(chars: List[str]) -> None:
    # TODO
    pass


# c3 = list("the sky is blue"); exercise_3_reverse_words_ii(c3); assert "".join(c3) == "blue is sky the"
# c3b = list("a"); exercise_3_reverse_words_ii(c3b); assert c3b == ["a"]

## Bài 4 — Biến thể: Rotate Array (LeetCode #189) bằng ba lần đảo

Xoay mảng sang phải `k` bước tại chỗ: đảo cả, đảo `k` phần tử đầu, đảo phần còn lại.

In [19]:
def exercise_4_rotate(nums: List[int], k: int) -> None:
    # TODO
    pass


# r4 = [1, 2, 3, 4, 5, 6, 7]; exercise_4_rotate(r4, 3); assert r4 == [5, 6, 7, 1, 2, 3, 4]
# r4b = [-1, -100, 3, 99]; exercise_4_rotate(r4b, 2); assert r4b == [3, 99, -1, -100]
# r4c = [1, 2]; exercise_4_rotate(r4c, 5); assert r4c == [2, 1]

## Bài 5 — Mini challenge: đảo thứ tự các từ nhưng giữ nguyên **các khối khoảng trắng**

Chuỗi xen kẽ khối khoảng trắng / từ. Giữ nguyên dãy các khối khoảng trắng (độ dài, thứ tự, cả đầu/cuối); chỉ các **từ** đổi thứ tự vào các ô từ.

```python
f("x  abc d") -> "d  abc x"
```

```python
f("  hello   big world ") -> "  world   big hello "
f("a b") -> "b a"
```

In [20]:
def exercise_5_reverse_keep_spaces(s: str) -> str:
    # TODO
    pass


# assert exercise_5_reverse_keep_spaces("  hello   big world ") == "  world   big hello "
# assert exercise_5_reverse_keep_spaces("a b") == "b a"
# assert exercise_5_reverse_keep_spaces("   ") == "   "
# assert exercise_5_reverse_keep_spaces("x  abc d") == "d  abc x"

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_inplace` sau khi tự làm.

**Bài 3**
- Mức 1: giống `reverse_words_inplace` nhưng bỏ bước dọn.

**Bài 4**
- Mức 1: `k %= n` trước.

**Bài 5**
- Mức 1: tách chuỗi thành các **token** xen kẽ (từ / khối khoảng trắng), ví dụ bằng `re.findall(r"\S+|\s+", s)`.
- Mức 2: lấy danh sách từ đảo ngược, rồi duyệt token: token là từ → thay bằng từ kế tiếp trong danh sách đảo.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
_ = trace_inplace("  Bob    Loves  Alice   ")

ban đầu    : '  Bob    Loves  Alice   '
sau dọn    : 'Bob Loves Alice'  (n = 15)
đảo cả     : 'ecilA sevoL boB'
đảo [0:5] : 'Alice sevoL boB'
đảo [6:11] : 'Alice Loves boB'
đảo [12:15] : 'Alice Loves Bob'


## Đáp án Bài 2

In [22]:
def answer_2_reverse_words(s: str) -> str:
    return Solution().reverseWords(s)

assert answer_2_reverse_words("the sky is blue") == "blue is sky the"
assert answer_2_reverse_words("  hello world  ") == "world hello"
assert answer_2_reverse_words("a good   example") == "example good a"
print("Bài 2 PASS (giống Solution ở mục 8)")

Bài 2 PASS (giống Solution ở mục 8)


## Đáp án Bài 3

In [23]:
def answer_3_reverse_words_ii(chars: List[str]) -> None:
    n = len(chars)
    reverse_range(chars, 0, n - 1)
    start = 0
    for k in range(n + 1):
        if k == n or chars[k] == " ":
            reverse_range(chars, start, k - 1)
            start = k + 1

c3 = list("the sky is blue"); answer_3_reverse_words_ii(c3); assert "".join(c3) == "blue is sky the"
c3b = list("a"); answer_3_reverse_words_ii(c3b); assert c3b == ["a"]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [24]:
def answer_4_rotate(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n
    reverse_range(nums, 0, n - 1)
    reverse_range(nums, 0, k - 1)
    reverse_range(nums, k, n - 1)

r4 = [1, 2, 3, 4, 5, 6, 7]; answer_4_rotate(r4, 3); assert r4 == [5, 6, 7, 1, 2, 3, 4]
r4b = [-1, -100, 3, 99]; answer_4_rotate(r4b, 2); assert r4b == [3, 99, -1, -100]
r4c = [1, 2]; answer_4_rotate(r4c, 5); assert r4c == [2, 1]
for _ in range(2000):
    a = [random.randint(0, 9) for _ in range(random.randint(1, 10))]
    k = random.randint(0, 25)
    b = a[:]; answer_4_rotate(b, k)
    assert b == a[-(k % len(a)):] + a[:-(k % len(a))] if k % len(a) else b == a
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [25]:
import re

def answer_5_reverse_keep_spaces(s: str) -> str:
    tokens = re.findall(r"\S+|\s+", s)
    words = [t for t in tokens if not t.isspace()][::-1]
    out, k = [], 0
    for t in tokens:
        if t.isspace():
            out.append(t)
        else:
            out.append(words[k]); k += 1
    return "".join(out)

assert answer_5_reverse_keep_spaces("  hello   big world ") == "  world   big hello "
assert answer_5_reverse_keep_spaces("a b") == "b a"
assert answer_5_reverse_keep_spaces("   ") == "   "
assert answer_5_reverse_keep_spaces("x  abc d") == "d  abc x"
for _ in range(2000):
    s = "".join(random.choice("ab ") for _ in range(random.randint(0, 15)))
    r = answer_5_reverse_keep_spaces(s)
    assert re.findall(r"\s+", r) == re.findall(r"\s+", s)                  # cùng dãy khối khoảng trắng
    assert (r[:1] == " ") == (s[:1] == " ") and (r[-1:] == " ") == (s[-1:] == " ")
    assert r.split() == s.split()[::-1]
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Python: `" ".join(reversed(s.split()))` (nhớ `split()` không tham số). Tự làm: duyệt từ cuối, bỏ dấu cách, cắt từng từ. Tại chỗ `O(1)`: dọn khoảng trắng bằng con trỏ ghi → **đảo cả chuỗi** → **đảo từng từ**.

---

## Checklist trước khi nộp

- [ ] Dùng `split()` chứ không `split(" ")`?
- [ ] Không có dấu cách thừa ở đầu, cuối, giữa?
- [ ] Bản tại chỗ: bỏ dấu cách cuối sau khi dọn?
- [ ] Đảo **cả** chuỗi **và** từng từ?